In [1]:
import numpy as np

In [2]:
# from pathlib import Path
# import numpy as np

# import jax
# import jax.numpy as jnp
# from flax.serialization import from_bytes

# from models import create_model
# from data_generation import generate_dataset


# # =============================================================================
# # Configuration
# # =============================================================================

# NODES = 10
# DIMENSIONS = 2

# MODEL_NAME = "HNN"
# HIDDEN_DIM = 1000

# INPUT_DIMS = 2 * NODES * (NODES - 1)
# OUTPUT_DIMS = 2 ** NODES

# PARAMS_PATH = Path(
#     "Models_26_05_26/run_HNN_10_MAE_5M_2/params.msgpack"
# )

# N_SAMPLES = 1000
# GEN_BATCH_SIZE = 1000
# SEED = 123

# # Important:
# # False because your model was trained on unnormalised amplitudes.
# NORMED_DATA = False


# # =============================================================================
# # Generate fresh test samples
# # =============================================================================

# print("Generating fresh test samples...")

# generated = generate_dataset(
#     vertices=NODES,
#     dimensions=DIMENSIONS,
#     n_samples=N_SAMPLES,
#     batch_size=GEN_BATCH_SIZE,
#     seed=SEED,
#     save_to_file=False,
#     normed_data=NORMED_DATA,
# )

# # Some versions return (X, Y), some may return (X, Y, levels)
# if len(generated) == 2:
#     X_test, Y_test = generated
# elif len(generated) == 3:
#     X_test, Y_test, levels = generated
# else:
#     raise ValueError(f"Unexpected generate_dataset output length: {len(generated)}")

# X_test = np.asarray(X_test, dtype=np.float32)
# Y_test = np.asarray(Y_test, dtype=np.float32)

# print("X_test:", X_test.shape)
# print("Y_test:", Y_test.shape)

# if X_test.shape != (N_SAMPLES, INPUT_DIMS):
#     raise ValueError(f"Wrong X shape: {X_test.shape}, expected {(N_SAMPLES, INPUT_DIMS)}")

# if Y_test.shape != (N_SAMPLES, OUTPUT_DIMS):
#     raise ValueError(f"Wrong Y shape: {Y_test.shape}, expected {(N_SAMPLES, OUTPUT_DIMS)}")


# # =============================================================================
# # Load model and params
# # =============================================================================

# print("\nLoading model...")

# model = create_model(
#     model_name=MODEL_NAME,
#     hidden_dims=HIDDEN_DIM,
#     out_dims=OUTPUT_DIMS,
#     nodes=NODES,
# )

# key = jax.random.PRNGKey(0)
# dummy_x = jnp.zeros((1, INPUT_DIMS), dtype=jnp.float32)

# params_template = model.init(key, dummy_x)

# with open(PARAMS_PATH, "rb") as f:
#     params = from_bytes(params_template, f.read())

# print("Loaded params from:", PARAMS_PATH)


# # =============================================================================
# # Predict
# # =============================================================================

# print("\nRunning prediction...")

# Xb = jax.device_put(jnp.asarray(X_test, dtype=jnp.float32))

# Y_pred = model.apply(params, Xb)
# jax.block_until_ready(Y_pred)

# Y_pred = np.asarray(Y_pred)
# Y_true = np.asarray(Y_test)


# # =============================================================================
# # Metrics
# # =============================================================================

# def normalized_fidelity_np(y_pred, y_true, eps=1e-12):
#     y_pred_n = y_pred / (np.linalg.norm(y_pred, axis=1, keepdims=True) + eps)
#     y_true_n = y_true / (np.linalg.norm(y_true, axis=1, keepdims=True) + eps)

#     overlap = np.sum(y_pred_n * y_true_n, axis=1)
#     fid = overlap ** 2
#     return fid

# print(Y_pred[0:5])
# print(Y_true[0:5])
# mae = np.mean(np.abs(Y_pred - Y_true))
# mse = np.mean((Y_pred - Y_true) ** 2)

# zero_mae = np.mean(np.abs(Y_true))
# mean_pred = np.mean(Y_true, axis=0, keepdims=True)
# mean_mae = np.mean(np.abs(mean_pred - Y_true))

# fid = normalized_fidelity_np(Y_pred, Y_true)


# # =============================================================================
# # Print scale comparison
# # =============================================================================

# print("\n================ Target scale ================")
# print("Target mean(abs):", np.mean(np.abs(Y_true)))
# print("Target median(abs):", np.median(np.abs(Y_true)))
# print("Target std:", np.std(Y_true))
# print("Target max(abs):", np.max(np.abs(Y_true)))
# print("Target min(abs):", np.min(np.abs(Y_true)))

# target_norms = np.linalg.norm(Y_true, axis=1)

# print("\nTarget vector norms")
# print("mean norm:", target_norms.mean())
# print("median norm:", np.median(target_norms))
# print("std norm:", target_norms.std())
# print("min norm:", target_norms.min())
# print("max norm:", target_norms.max())


# print("\n================ Prediction scale ================")
# print("Pred mean(abs):", np.mean(np.abs(Y_pred)))
# print("Pred median(abs):", np.median(np.abs(Y_pred)))
# print("Pred std:", np.std(Y_pred))
# print("Pred max(abs):", np.max(np.abs(Y_pred)))
# print("Pred min(abs):", np.min(np.abs(Y_pred)))

# pred_norms = np.linalg.norm(Y_pred, axis=1)

# print("\nPrediction vector norms")
# print("mean norm:", pred_norms.mean())
# print("median norm:", np.median(pred_norms))
# print("std norm:", pred_norms.std())
# print("min norm:", pred_norms.min())
# print("max norm:", pred_norms.max())


# print("\n================ Baselines and model ================")
# print("Model MAE:", mae)
# print("Model MSE:", mse)
# print("Zero predictor MAE:", zero_mae)
# print("Mean predictor MAE:", mean_mae)

# print("\nImprovement over zero predictor:")
# print("absolute:", zero_mae - mae)
# print("relative:", (zero_mae - mae) / zero_mae)


# print("\n================ Normalized fidelity ================")
# print("Mean fidelity:", fid.mean())
# print("Median fidelity:", np.median(fid))
# print("Std fidelity:", fid.std())
# print("Min fidelity:", fid.min())
# print("Max fidelity:", fid.max())


# # =============================================================================
# # Diagnosis
# # =============================================================================

# print("\n================ Diagnosis ================")

# target_mean_abs = np.mean(np.abs(Y_true))
# pred_mean_abs = np.mean(np.abs(Y_pred))

# if pred_mean_abs < 0.1 * target_mean_abs:
#     print("Diagnosis: prediction scale is near zero compared to target.")
# elif pred_mean_abs < 0.5 * target_mean_abs:
#     print("Diagnosis: prediction scale is much smaller than target.")
# elif 0.5 * target_mean_abs <= pred_mean_abs <= 2.0 * target_mean_abs:
#     print("Diagnosis: prediction scale is comparable to target.")
# else:
#     print("Diagnosis: prediction scale is much larger than target.")

# if mae >= 0.95 * zero_mae:
#     print("Model is almost equivalent to zero predictor.")
# elif mae >= 0.80 * zero_mae:
#     print("Model is only weakly better than zero predictor.")
# else:
#     print("Model is clearly better than zero predictor by MAE.")

# if fid.mean() < 0.1:
#     print("Fidelity is very low: state direction is not learned well.")
# elif fid.mean() < 0.5:
#     print("Fidelity is moderate/weak.")
# else:
#     print("Fidelity is reasonably high.")

In [3]:
import numpy as np

DATA_PATH = "/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean/data/node10_test/data_00003.npz"

with np.load(DATA_PATH) as d:
    Y = d["amps"][:100]
    X = d["weights"][:100]

print("X shape:", X.shape)
print("Y shape:", Y.shape)

print("\nAmplitude scale:")
print("mean(abs(Y)):", np.mean(np.abs(Y)))
print("median(abs(Y)):", np.median(np.abs(Y)))
print("std(abs(Y)):", np.std(np.abs(Y)))
print("max(abs(Y)):", np.max(np.abs(Y)))
print("min(abs(Y)):", np.min(np.abs(Y)))

norms = np.linalg.norm(Y, axis=1)

print("\nVector norms:")
print("mean norm:", norms.mean())
print("median norm:", np.median(norms))
print("std norm:", norms.std())
print("min norm:", norms.min())
print("max norm:", norms.max())

zero_mae = np.mean(np.abs(Y))
mean_pred = np.mean(Y, axis=0, keepdims=True)
mean_mae = np.mean(np.abs(mean_pred - Y))

print("\nBaselines:")
print("Zero predictor MAE:", zero_mae)
print("Mean predictor MAE:", mean_mae)

KeyboardInterrupt: 

# Optimiser quick-test notebook

Use this notebook to test the cleaned optimiser files step by step before running a full optimisation job.

Expected files in the same repo folder:

```text
optimiser.py
optimisation_utils.py
target_states.py
models.py
data_generation.py
data_generation_utils.py
```

The notebook does **not** start a full expensive run by default. It checks imports, target states, initial samples, model loading, PyTheus catalog, and one optimisation step.

In [4]:
from pathlib import Path
import sys
import numpy as np
import jax
import jax.numpy as jnp

# EDIT THIS if the notebook is outside your repo folder
REPO_DIR = Path.cwd()

# Example:
# REPO_DIR = Path('/home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean')

sys.path.insert(0, str(REPO_DIR))

print('Repo dir:', REPO_DIR)
print('JAX devices:', jax.devices())
print('Default backend:', jax.default_backend())

Repo dir: /home/bo48god/projects/deep_dreaming/surrogate_model_clean_all_codes_with_config/surrogate_model_clean


ERROR:2026-05-26 15:01:27,012:jax._src.xla_bridge:475: Jax plugin configuration error: Exception when calling jax_plugins.xla_cuda12.initialize()
Traceback (most recent call last):
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax/_src/xla_bridge.py", line 473, in discover_pjrt_plugins
    plugin_module.initialize()
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 328, in initialize
    _check_cuda_versions(raise_on_first_error=True)
  File "/home/bo48god/projects/deep_dreaming/env_jax/lib/python3.12/site-packages/jax_plugins/xla_cuda12/__init__.py", line 285, in _check_cuda_versions
    local_device_count = cuda_versions.cuda_device_count()
                         ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
RuntimeError: jaxlib/cuda/versions_helpers.cc:113: operation cuInit(0) failed: Unknown CUDA error 303; cuGetErrorName failed. This probably means that JAX was unable to load the CU

JAX devices: [CpuDevice(id=0)]
Default backend: cpu


## 1. Import optimiser modules

In [5]:
from target_states import get_target_state
from optimisation_utils import (
    prepare_input_data,
    setup_pytheus_catalog,
    load_trained_model,
    build_optimizer,
    build_forward_metrics,
    build_dream_step,
    pytheus_state_from_x,
    fidelity_np,
)

print('Imports OK')

Imports OK


## 2. Define a small test config

Edit this cell. For a first test, use `NPHOTONS = 4` and a small conditioned dataset or generated data.

For testing only target states / PyTheus, the model path does not need to exist. For model loading and optimisation step, `MODEL_PATH` must point to a valid `params.msgpack`.

In [22]:
NPHOTONS = 4
TARGET_NAME = 'GHZ'  # 'GHZ', 'W', 'LINEAR_CLUSTER', 'SINGLE', 'ZERO'
MODEL_TYPE = 'HNN'  # 'HNN' or 'FNN'

# HNN: integer architecture. FNN: tuple, e.g. (128, 128).
ARCHITECTURE = 400

# Set this to your trained params file before model-loading test.
MODEL_PATH = '/home/bo48god/projects/deep_dreaming/src/projects/src/HNN_Nodes4_06_05_26/run_HNN_4_MAE_400_20M_cosine_decay_gputest/params.msgpack'

# Mode A: generate initial samples using data_generation.generate_dataset().
GENERATE_DATA = True

# Mode B: load existing initial samples from .npz with keys weights, amps, optional levels.
CONDITIONED_DATA_PATH = '/path/to/conditioned_dataset.npz'

CFG_TEST = {
    'description': 'notebook optimiser quick test',
    'n': NPHOTONS,
    'dimensions': 2,
    'target_name': TARGET_NAME,
    'model_type': MODEL_TYPE,

    # initial data
    'generate_data': GENERATE_DATA,
    'conditioned_data_path': CONDITIONED_DATA_PATH,
    'max_initial_samples': 5,
    'data_samples': 5,
    'data_batch_size': 5,
    'data_seed': 4,
    'low_fidelity_threshold': 0.999,  # high threshold keeps almost everything for quick test
    'save_generated_data': False,
    'data_out_dir': 'data/optimiser_notebook_test',
    'normed_data': True,
    'generation_gpu_batch_size': 5,
    'data_shard_size': 5,

    # model
    'architecture': ARCHITECTURE,
    'model_path': MODEL_PATH,
    'normalize_model_output': True,
    'input_dim': 2 * NPHOTONS * (NPHOTONS - 1),
    'out_dim': 2 ** NPHOTONS,

    # optimisation
    'learning_rate': 1e-2,
    'min_learning_rate': 1e-6,
    'lr_decay_steps': 100,
    'lr_exponent': 1.0,
    'num_steps': 10,
    'print_every': 1,
    'verify_every': 1,
    'early_stop_nn_fid': 0.9999,
    'lambda_l1': 1e-3,
    'seed': 46,

    # output
    'results_root': 'optimiser_notebook_results',
    'folder_name': 'quick_test_2',
    'date': 'notebook',
    'k_value': 0,
    'noise': 0,
    'CPU_GPU': 'GPU',
}

CFG_TEST

{'description': 'notebook optimiser quick test',
 'n': 4,
 'dimensions': 2,
 'target_name': 'GHZ',
 'model_type': 'HNN',
 'generate_data': True,
 'conditioned_data_path': '/path/to/conditioned_dataset.npz',
 'max_initial_samples': 5,
 'data_samples': 5,
 'data_batch_size': 5,
 'data_seed': 4,
 'low_fidelity_threshold': 0.999,
 'save_generated_data': False,
 'data_out_dir': 'data/optimiser_notebook_test',
 'normed_data': True,
 'generation_gpu_batch_size': 5,
 'data_shard_size': 5,
 'architecture': 400,
 'model_path': '/home/bo48god/projects/deep_dreaming/src/projects/src/HNN_Nodes4_06_05_26/run_HNN_4_MAE_400_20M_cosine_decay_gputest/params.msgpack',
 'normalize_model_output': True,
 'input_dim': 24,
 'out_dim': 16,
 'learning_rate': 0.01,
 'min_learning_rate': 1e-06,
 'lr_decay_steps': 100,
 'lr_exponent': 1.0,
 'num_steps': 10,
 'print_every': 1,
 'verify_every': 1,
 'early_stop_nn_fid': 0.9999,
 'lambda_l1': 0.001,
 'seed': 46,
 'results_root': 'optimiser_notebook_results',
 'folder_

## 3. Test target state

In [23]:
target = get_target_state(CFG_TEST['target_name'], CFG_TEST['n'])
print('target shape:', target.shape)
print('target norm:', np.linalg.norm(target))
print('nonzero entries:', np.nonzero(np.abs(target) > 1e-12)[0][:20])

target shape: (16,)
target norm: 0.99999994
nonzero entries: [ 0 15]


## 4. Test initial sample preparation

This either generates a few samples or loads a small `.npz`, depending on `CFG_TEST['generate_data']`.

In [24]:
data = prepare_input_data(CFG_TEST)
print('number of groups:', len(data))

levels, X, Y = data[0]
print('levels:', levels.shape, levels[:10])
print('X:', X.shape, X.dtype)
print('Y:', Y.shape, Y.dtype)
print('first Y norm:', np.linalg.norm(Y[0]))

2026-05-26 15:06:36,282 | INFO | === Random Dense Graph Dataset Generation ===
2026-05-26 15:06:36,285 | INFO | JAX devices: [CpuDevice(id=0)]
2026-05-26 15:06:36,286 | INFO | vertices=4, dimensions=2
2026-05-26 15:06:36,288 | INFO | n_samples=5, batch_size=5
2026-05-26 15:06:36,289 | INFO | seed=4
2026-05-26 15:06:36,291 | INFO | normed_data=True
2026-05-26 15:06:36,293 | INFO | graph regime=Level 5 random dense weights in [-1, 1]
2026-05-26 15:06:36,295 | INFO | Precomputing PyTheus perfect-matching catalog...
2026-05-26 15:06:36,298 | INFO | num_edges=24
2026-05-26 15:06:36,299 | INFO | n_kets=16
2026-05-26 15:06:36,301 | INFO | catalog tensor shape=(16, 3, 2)
2026-05-26 15:06:36,302 | INFO | catalog time=0.01s
2026-05-26 15:06:36,303 | INFO | Transferring catalog to GPU...
2026-05-26 15:06:36,306 | INFO | JIT warmup...
2026-05-26 15:06:36,308 | INFO | Generating data batches...
2026-05-26 15:06:36,311 | INFO | batch     1 |          5/5 (100.0%) | 0.0s | 3,305 samples/s
2026-05-26 

## 5. Test PyTheus catalog and exact-state computation

In [25]:
edges, tensor, mask = setup_pytheus_catalog(CFG_TEST['n'])
tensor_jax = jnp.asarray(tensor, dtype=jnp.int32)
mask_jax = jnp.asarray(mask, dtype=jnp.float32)

print('num edges:', len(edges))
print('tensor shape:', tensor.shape)
print('mask shape:', mask.shape)

py_state, _ = pytheus_state_from_x(X[0], tensor_jax, mask_jax)
print('PyTheus state shape:', py_state.shape)
print('PyTheus state norm:', np.linalg.norm(py_state))
print('PyTheus fidelity to target:', fidelity_np(py_state, target))

num edges: 24
tensor shape: (16, 3, 2)
mask shape: (16, 3)
PyTheus state shape: (16,)
PyTheus state norm: 1.0
PyTheus fidelity to target: 0.06095704436302185


## 6. Test model loading

Run this only after `MODEL_PATH` points to a real `params.msgpack` compatible with `NPHOTONS`, `MODEL_TYPE`, and `ARCHITECTURE`.

In [26]:
model_path = Path(CFG_TEST['model_path'])
print('Model path exists:', model_path.exists(), model_path)

if model_path.exists():
    apply_fn, params = load_trained_model(CFG_TEST)
    y_pred = apply_fn(params, jnp.asarray(X[0], dtype=jnp.float32))
    y_pred = np.asarray(jax.block_until_ready(y_pred))
    print('prediction shape:', y_pred.shape)
    print('prediction norm:', np.linalg.norm(y_pred))
else:
    print(F'Skipping model loading. Set {CFG_TEST["model_path"]} first.')

Model path exists: True /home/bo48god/projects/deep_dreaming/src/projects/src/HNN_Nodes4_06_05_26/run_HNN_4_MAE_400_20M_cosine_decay_gputest/params.msgpack
prediction shape: (16,)
prediction norm: 2.709554


## 7. Test one optimisation step

This checks that the loss/gradient/update path works. It runs only if the model path exists.

In [27]:
if Path(CFG_TEST['model_path']).exists():
    apply_fn, params = load_trained_model(CFG_TEST)
    optimizer, lr_schedule = build_optimizer(CFG_TEST)
    forward_metrics = build_forward_metrics(apply_fn, params, CFG_TEST)
    dream_step = build_dream_step(forward_metrics, optimizer)

    x = jnp.asarray(X[0], dtype=jnp.float32)
    y_target = jnp.asarray(target, dtype=jnp.float32)
    opt_state = optimizer.init(x)

    loss0, pred0, fid0, mae0, l10 = forward_metrics(x, y_target)
    print('Before step:')
    print('loss:', float(loss0), 'fid:', float(fid0), 'mae:', float(mae0), 'l1:', float(l10))

    x, opt_state, loss1, pred1, fid1, mae1, l11, grads, raw_update, update_before_clip, update_after_clip = dream_step(
        x, opt_state, y_target
    )

    jax.block_until_ready(loss1)
    print('After one step:')
    print('loss:', float(loss1), 'fid:', float(fid1), 'mae:', float(mae1), 'l1:', float(l11))
    print('grad norm:', float(jnp.linalg.norm(grads)))
    print('x range:', float(jnp.min(x)), float(jnp.max(x)))
else:
    print(F'Skipping optimisation-step test. Set {CFG_TEST["model_path"]} first.')
    

Before step:
loss: 0.9520895481109619 fid: 0.06097165122628212 mae: 0.2409074306488037 l1: 0.01306119654327631
After one step:
loss: 0.9434854984283447 fid: 0.06953569501638412 mae: 0.23779264092445374 l1: 0.01302119717001915
grad norm: 0.22936594486236572
x range: -0.8483278751373291 0.958306074142456


## 8. Run the full optimiser on the test config

This imports `run_optimisation()` from `optimiser.py` and runs it with `CFG_TEST`. Keep `data_samples` and `num_steps` small first.

In [28]:
# Uncomment only after the cells above work.
from optimiser import run_optimisation
result_dir = run_optimisation(CFG_TEST)
print('Results:', result_dir)

2026-05-26 15:06:40,179 | INFO | === Random Dense Graph Dataset Generation ===
2026-05-26 15:06:40,182 | INFO | JAX devices: [CpuDevice(id=0)]
2026-05-26 15:06:40,183 | INFO | vertices=4, dimensions=2
2026-05-26 15:06:40,184 | INFO | n_samples=5, batch_size=5
2026-05-26 15:06:40,186 | INFO | seed=4
2026-05-26 15:06:40,187 | INFO | normed_data=True
2026-05-26 15:06:40,190 | INFO | graph regime=Level 5 random dense weights in [-1, 1]
2026-05-26 15:06:40,191 | INFO | Precomputing PyTheus perfect-matching catalog...
2026-05-26 15:06:40,194 | INFO | num_edges=24
2026-05-26 15:06:40,196 | INFO | n_kets=16
2026-05-26 15:06:40,197 | INFO | catalog tensor shape=(16, 3, 2)
2026-05-26 15:06:40,198 | INFO | catalog time=0.01s
2026-05-26 15:06:40,199 | INFO | Transferring catalog to GPU...
2026-05-26 15:06:40,203 | INFO | JIT warmup...
2026-05-26 15:06:40,206 | INFO | Generating data batches...
2026-05-26 15:06:40,209 | INFO | batch     1 |          5/5 (100.0%) | 0.0s | 4,614 samples/s
2026-05-26 

## Recommended quick-test sequence

1. Run cells 1–5 with `NPHOTONS = 4` and `GENERATE_DATA = True`.
2. Set `MODEL_PATH` to a compatible small trained model and run cells 6–7.
3. Run cell 8 with `data_samples = 2` and `num_steps = 10`.
4. Only then increase to your real `n = 8/10`, target, model path, and sample count.